# Pharmasense — TFT Training
**Temporal Fusion Transformer for baseline demand + endemic disease forecasting**

## What this model covers
- **Epidemiological diseases:** `acute_diarrhoeal_disease` — endemic/seasonal, not spatially driven by adjacency (that's ST-GNN)
- **Why ADD here:** ADD outbreaks are caused by contaminated water/food, not vector spread. They follow rainfall/seasonal patterns. Pure time-series — TFT is the right tool.
- **Why TFT over ARIMA/Prophet:** TFT handles multiple covariates (season, location, disease type), learns variable selection, and produces *quantile outputs* — giving us uncertainty bounds for procurement planning (e.g. 'worst-case 90th percentile demand = stock this much').
- **Output:** `tft_best.ckpt` + `tft_latest.pt` saved to Drive + auto-downloaded

## Before running
1. **Runtime → Change runtime type → T4 GPU**
2. **Add Colab Secret:** click 🔑 in left sidebar → `PHARMASENSE_DB_PASSWORD`
3. **Runtime → Run All**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/02_train_tft.ipynb)

In [ ]:
# ── 0a. Check GPU ──────────────────────────────────────────────────────────────
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f'GPU: {r.stdout.strip()}')
else:
    raise RuntimeError('No GPU. Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 0b. Install dependencies ───────────────────────────────────────────────────
# pytorch-forecasting bundles compatible torch + lightning versions
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pytorch-forecasting==1.0.0',
    'pytorch-lightning==2.2.4',
    'psycopg2-binary', 'pandas', 'numpy', 'scikit-learn', 'matplotlib'], check=True)
print('Dependencies installed.')

In [ ]:
# ── 1. Config ──────────────────────────────────────────────────────────────────
from google.colab import userdata

DB_CONFIG = dict(
    host     = 'db.xwtvujwdybvjbaucshbt.supabase.co',
    port     = 5432,
    dbname   = 'postgres',
    user     = 'postgres',
    password = userdata.get('PHARMASENSE_DB_PASSWORD'),
    sslmode  = 'require',
)

# ── TFT scope: ADD (endemic) + all diseases for demand baseline ─────────────────
# Primary: ADD (acute diarrhoeal disease) — seasonal, not adjacency-driven
# Secondary: dengue + malaria + cholera baseline (non-outbreak steady-state trend)
# ST-GNN handles spatial outbreak dynamics; TFT handles temporal demand baseline
TFT_DISEASES = ['acute_diarrhoeal_disease', 'dengue', 'malaria', 'cholera']

# ── TFT hyperparameters ───────────────────────────────────────────────────────
MAX_ENCODER_LEN     = 24   # 6 months of history (same as ST-GNN lookback for consistency)
MAX_PREDICTION_LEN  = 4    # forecast 4 weeks (1 month) — longer horizon than ST-GNN
MIN_OBS_PER_GROUP   = 30   # drop (district, disease) series with fewer than 30 weeks of data
BATCH_SIZE          = 64
MAX_EPOCHS          = 60
LEARNING_RATE       = 3e-3
HIDDEN_SIZE         = 32   # TFT hidden state (small = fast, adequate for our dataset)
ATTENTION_HEADS     = 4    # multi-head attention (4 heads × 8 dims = 32 total)
DROPOUT             = 0.1  # low — TFT has internal gating, not much extra dropout needed
HIDDEN_CONT_SIZE    = 16   # continuous variable processing size

DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'

import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'TFT diseases: {TFT_DISEASES}')
print(f'Encoder: {MAX_ENCODER_LEN}w | Horizon: {MAX_PREDICTION_LEN}w | Min obs: {MIN_OBS_PER_GROUP}w')

In [ ]:
# ── 2. Mount Drive ─────────────────────────────────────────────────────────────
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive ready: {DRIVE_OUTPUT}')

In [ ]:
# ── 3. Load + prepare data ─────────────────────────────────────────────────────
import psycopg2
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

conn = psycopg2.connect(**DB_CONFIG)

# Districts — static covariates (location, state)
dist_df = pd.read_sql("""
    SELECT id, name, state, centroid_lat, centroid_lon FROM districts;
""", conn)

# All 4 diseases for TFT
dis_filter = "', '".join(TFT_DISEASES)
epid_df = pd.read_sql(f"""
    SELECT district_id, disease, week_start_date, cases
    FROM epid_weekly
    WHERE disease IN ('{dis_filter}')
    ORDER BY week_start_date, district_id;
""", conn)
conn.close()

epid_df['week_start_date'] = pd.to_datetime(epid_df['week_start_date'])
epid_df['log_cases']       = np.log1p(epid_df['cases'].clip(lower=0).astype(float))

# Merge static district features
epid_df = epid_df.merge(dist_df[['id','state','centroid_lat','centroid_lon']],
                        left_on='district_id', right_on='id', how='left').drop('id', axis=1)

print(f'Records loaded: {len(epid_df):,}')
print(epid_df.groupby('disease')[['cases','log_cases']].agg(['count','mean']).round(3))

In [ ]:
# ── 4. Feature engineering + TimeSeriesDataSet ────────────────────────────────
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

# Categorical encoders
le_district = LabelEncoder()
le_state    = LabelEncoder()
le_disease  = LabelEncoder()
epid_df['district_enc'] = le_district.fit_transform(epid_df['district_id'].astype(str))
epid_df['state_enc']    = le_state.fit_transform(epid_df['state'].fillna('Unknown').astype(str))
epid_df['disease_enc']  = le_disease.fit_transform(epid_df['disease'].astype(str))

# Time index (integer weeks from epoch)
min_date            = epid_df['week_start_date'].min()
epid_df['time_idx'] = ((epid_df['week_start_date'] - min_date).dt.days // 7).astype(int)

# Month-of-year as known time-varying covariate (seasonal signal)
epid_df['month'] = epid_df['week_start_date'].dt.month.astype(float)

# Group key: one series per (district, disease)
epid_df['group_id'] = epid_df['district_id'].astype(str) + '__' + epid_df['disease']

# Drop groups with insufficient history
min_len      = MAX_ENCODER_LEN + MAX_PREDICTION_LEN
gc           = epid_df.groupby('group_id')['time_idx'].count()
valid_groups = gc[gc >= min_len].index
epid_df      = epid_df[epid_df['group_id'].isin(valid_groups)].copy()

print(f'Records after min-obs filter: {len(epid_df):,}')
print(f'Valid (district × disease) series: {epid_df["group_id"].nunique()}')

# Time split: 80% train, 20% val
max_t_idx    = epid_df['time_idx'].max()
train_cutoff = int(max_t_idx * 0.80)

training = TimeSeriesDataSet(
    epid_df[epid_df['time_idx'] <= train_cutoff],
    time_idx                  = 'time_idx',
    target                    = 'log_cases',
    group_ids                 = ['group_id'],
    max_encoder_length        = MAX_ENCODER_LEN,
    max_prediction_length     = MAX_PREDICTION_LEN,
    # Static categoricals: things that don't change per series
    static_categoricals       = ['district_enc', 'state_enc', 'disease_enc'],
    # Static reals: fixed per group
    static_reals              = ['centroid_lat', 'centroid_lon'],
    # Known time-varying: available for both past and future (e.g. calendar features)
    time_varying_known_reals  = ['time_idx', 'month'],
    # Unknown time-varying: the target + other observed-but-not-future-known vars
    time_varying_unknown_reals = ['log_cases'],
    target_normalizer         = GroupNormalizer(groups=['group_id'], transformation='log1p'),
    allow_missing_timesteps   = True,
    add_relative_time_idx     = True,   # adds (t - t_pred) / max_encoder as feature
    add_target_scales         = True,   # adds learned scale per group
)

validation = TimeSeriesDataSet.from_dataset(training, epid_df, predict=True, stop_randomization=True)

from torch.utils.data import DataLoader
train_dl = training.to_dataloader(train=True,  batch_size=BATCH_SIZE,     num_workers=2)
val_dl   = validation.to_dataloader(train=False, batch_size=BATCH_SIZE*2, num_workers=2)

print(f'\nTraining samples: {len(training):,}')
print(f'Validation samples: {len(validation):,}')

In [ ]:
# ── 5. Define TFT model ────────────────────────────────────────────────────────
# TFT design notes:
# - output_size=7: 7 quantiles [0.02,0.1,0.25,0.5,0.75,0.9,0.98]
#   0.5 = median forecast (demand planning baseline)
#   0.9/0.98 = worst-case procurement buffer (don't stock-out)
#   0.02/0.1 = lower bound (avoid overstock wastage)
# - Variable selection networks inside TFT automatically learn which covariates
#   matter per group (disease-specific feature importance)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate            = LEARNING_RATE,
    hidden_size              = HIDDEN_SIZE,
    attention_head_size      = ATTENTION_HEADS,
    dropout                  = DROPOUT,
    hidden_continuous_size   = HIDDEN_CONT_SIZE,
    output_size              = 7,       # 7-quantile output
    loss                     = QuantileLoss(),
    log_interval             = 20,
    reduce_on_plateau_patience = 5,
    optimizer                = 'adam',
)

total = sum(p.numel() for p in tft.parameters() if p.requires_grad)
print(f'TFT  |  params: {total:,}  |  hidden: {HIDDEN_SIZE}  |  heads: {ATTENTION_HEADS}')
print(f'Quantile outputs: {tft.loss.quantiles}')

In [ ]:
# ── 6. Train ───────────────────────────────────────────────────────────────────
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, mode='min', verbose=True),
    ModelCheckpoint(
        dirpath  = DRIVE_OUTPUT,
        filename = 'tft_best',
        monitor  = 'val_loss',
        mode     = 'min',
        save_top_k = 1,
        verbose  = True,
    ),
    LearningRateMonitor(logging_interval='epoch'),
]

trainer = pl.Trainer(
    max_epochs         = MAX_EPOCHS,
    accelerator        = 'gpu' if torch.cuda.is_available() else 'cpu',
    devices            = 1,
    gradient_clip_val  = 0.1,
    callbacks          = callbacks,
    enable_progress_bar = True,
    logger             = TensorBoardLogger(DRIVE_OUTPUT + 'tb_logs/', name='tft'),
)

trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
print(f'Best checkpoint: {callbacks[1].best_model_path}')
print(f'Best val loss:   {callbacks[1].best_model_score:.5f}')

In [ ]:
# ── 7. Evaluate + variable importance ─────────────────────────────────────────
import matplotlib.pyplot as plt

best_tft = TemporalFusionTransformer.load_from_checkpoint(callbacks[1].best_model_path)
best_tft.eval()

predictions = best_tft.predict(val_dl, mode='raw', return_x=True, trainer_kwargs={'accelerator': 'gpu'})

# Plot 4 example forecasts with quantile uncertainty bands
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
n_series = len(validation)
for i, ax in enumerate(axes.flat):
    idx = i * (n_series // 4)
    best_tft.plot_prediction(predictions.x, predictions.output, idx=idx, ax=ax, show_future_observed=True)
    ax.set_title(f'Series {idx}: {epid_df.iloc[idx]["group_id"] if idx < len(epid_df) else ""}')
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'tft_forecast_examples.png')
plt.show()

# TFT's built-in variable importance (which features matter most per disease)
raw_predictions, x = best_tft.predict(val_dl, mode='raw', return_x=True)
importance = best_tft.calculate_prediction_actual_by_variable(x, raw_predictions)
best_tft.plot_prediction_actual_by_variable(importance)
plt.savefig(DRIVE_OUTPUT + 'tft_variable_importance.png')
plt.show()

In [ ]:
# ── 8. Save artifact ───────────────────────────────────────────────────────────
import torch
from datetime import datetime

ckpt_path = callbacks[1].best_model_path

artifact = {
    'model_checkpoint_path': ckpt_path,
    'dataset_parameters':    training.get_parameters(),
    'training_config': {
        'tft_diseases':          TFT_DISEASES,
        'max_encoder_length':    MAX_ENCODER_LEN,
        'max_prediction_length': MAX_PREDICTION_LEN,
        'hidden_size':           HIDDEN_SIZE,
        'attention_heads':       ATTENTION_HEADS,
        'quantiles':             list(tft.loss.quantiles.numpy()),
        'note': 'Median (q=0.5) for baseline demand; q0.9/0.98 for procurement buffer'
    },
    'label_encoders': {'district': le_district, 'state': le_state, 'disease': le_disease},
    'best_val_loss': float(callbacks[1].best_model_score),
    'trained_at':    datetime.now().isoformat(),
}

meta_path = DRIVE_OUTPUT + 'tft_latest.pt'
torch.save(artifact, meta_path)
print(f'Metadata saved: {meta_path}')
print(f'Checkpoint:     {ckpt_path}')
print()
print('Copy both files to ml/artifacts/ in your local repo:')
print('  ml/artifacts/tft_latest.pt')
print('  ml/artifacts/tft_best.ckpt')

from google.colab import files
files.download(meta_path)
files.download(ckpt_path)
print('Downloads started.')